In [4]:
"""
RunSignUp API - Race results for a given race and result set.
Docs: https://runsignup.com/Api/race/:race_id/results/get-results/GET
"""

import requests
import os

API_KEY    = os.getenv("RUNSIGNUP_API_KEY",    "TkPIUG2GLK95zjIAONwAyv348fQQ6TtY")
API_SECRET = os.getenv("RUNSIGNUP_API_SECRET", "oNsju6QeN0UEVZ1JK612wyay0dC2jLJ9")
AFLT_TOKEN = "uL3LJsNhaU6SFxIRWfJFjA0K24o2fddK"

RACE_ID       = 32802
EVENT_ID      = 987494   # 2025 Farm Bureau Watermelon Classic 5k RUN
RESULT_SET_ID = 564071

In [5]:
def get_all_results(race_id, event_id, result_set_id=None):
    url = f"https://runsignup.com/Rest/race/{race_id}/results/get-results"
    all_results = []
    page = 1

    while True:
        params = {
            "api_key":    API_KEY,
            "api_secret": API_SECRET,
            "aflt_token": AFLT_TOKEN,
            "format":     "json",
            "event_id":   event_id,
            "page":       page,
            "results_per_page": 100,
        }
        if result_set_id:
            params["individual_result_set_id"] = result_set_id

        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()

        page_results = []
        for rs in data.get("individual_results_sets", []):
            page_results.extend(rs.get("results", []))

        if not page_results:
            break

        all_results.extend(page_results)
        print(f"Page {page}: {len(page_results)} results")
        page += 1

    return all_results

In [6]:
results = get_all_results(RACE_ID, EVENT_ID, result_set_id=RESULT_SET_ID)
print(f"\nTotal: {len(results)} results")

for r in results[:5]:
    print(f"  {r.get('place'):>4}  {r.get('first_name')} {r.get('last_name'):<20}  {r.get('chip_time')}")

Page 1: 100 results
Page 2: 100 results
Page 3: 100 results
Page 4: 100 results
Page 5: 100 results
Page 6: 100 results
Page 7: 100 results
Page 8: 100 results
Page 9: 100 results
Page 10: 80 results

Total: 980 results
     1  Speed, Preston               16:21
     2  Porter, Will                  16:45
     3  Richardson, Slater                16:53
     4  Ward-McCammon, Dominic               17:02
     5  Gomez, Adam                  17:27


In [9]:
import pandas as pd

df = pd.DataFrame(results)

# Drop division placement columns (named like "division-7036465-placement")
df = df[[c for c in df.columns if not c.startswith("division-")]]

print(df.shape)
df.head(10)

(980, 14)


,result_id,place,bib,first_name,last_name,gender,city,state,country_code,clock_time,chip_time,pace,age,age_percentage
0,188309290,1,880,"Speed,",Preston,M,None,None,None,16:21,16:21,5:15,22,78.6%
1,188309291,2,405,"Porter,",Will,M,None,None,None,16:46,16:45,5:23,22,76.7%
2,188309292,3,337,"Richardson,",Slater,M,None,None,None,16:54,16:53,5:25,25,76.1%
3,188309293,4,460,"Ward-McCammon,",Dominic,M,None,None,None,17:03,17:02,5:28,22,75.4%
4,188309294,5,347,"Gomez,",Adam,M,None,None,None,17:27,17:27,5:36,23,73.6%
5,188309295,6,239,"Towery,",Walter,M,None,None,None,17:29,17:29,5:37,16,75.0%
6,188309296,7,815,"Montgomery,",Guage,M,None,None,None,17:32,17:32,5:38,23,73.3%
7,188309297,8,5,"Holiman,",Frank,M,None,None,None,17:46,17:46,5:42,30,72.3%
8,188309298,9,221,"Fulton,",Cooper,M,None,None,None,18:17,18:16,5:52,31,70.4%
9,188309299,10,917,"Tate,",R.T.,M,None,None,None,18:33,18:33,5:57,17,69.8%
